# Fine-Tuning with Feedback: Practice Exercise

Analyze multiple user feedback items to identify common themes, then build a meta-prompting system that synthesizes these critiques into an improved system prompt.

**What you'll implement:**
- An `analyze_feedback_themes` function that identifies common issues across multiple feedback items
- A `build_synthesis_prompt` function that creates a meta-prompt incorporating the identified themes
- A `generate_improved_prompt` function that uses the meta-prompt to create better instructions

**Estimated time:** 15-20 minutes

## Setup

Run this cell to import all required libraries and configure the environment.

In [1]:
# Setup - run this cell first

import os
from datetime import datetime
from typing import Dict, Any, List

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# Load environment variables
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")

print("Setup complete!")

Setup complete!


## Context

You are improving a customer support agent for a software company. The agent helps users troubleshoot issues with their product. After a week in production, you've collected feedback from multiple users - and patterns are emerging.

**Your task:** Rather than addressing each piece of feedback individually, you need to:
1. Analyze the feedback to identify **common themes** across users
2. Build a meta-prompt that **synthesizes** these themes into actionable improvements
3. Generate an improved system prompt that addresses the root issues

**Why this matters:** In production, you'll receive hundreds of feedback items. The skill isn't just writing meta-prompts - it's identifying which issues are systemic vs. one-off, and crafting improvements that address patterns rather than individual complaints.

## Provided Components

The support tools, agent creation, baseline prompt, and user feedback data are provided for you.

In [2]:
# Support tools - provided
@tool
def check_system_status(service: str) -> str:
    """Check the status of a system service.
    
    Args:
        service: Name of the service to check (api, database, auth, payments)
        
    Returns:
        Current status of the service
    """
    statuses = {
        "api": "Operational - Response time: 145ms",
        "database": "Operational - Connection pool: 45% utilized",
        "auth": "Degraded - Intermittent delays in token refresh",
        "payments": "Operational - All payment providers connected"
    }
    service_lower = service.lower().strip()
    return statuses.get(service_lower, f"Unknown service: {service}. Available: api, database, auth, payments")


@tool
def lookup_error_code(code: str) -> str:
    """Look up what an error code means and common solutions.
    
    Args:
        code: The error code to look up
        
    Returns:
        Description and solutions for the error
    """
    errors = {
        "E001": "Authentication failed. Solutions: 1) Check API key is valid, 2) Ensure key has required permissions, 3) Verify key hasn't expired",
        "E002": "Rate limit exceeded. Solutions: 1) Implement exponential backoff, 2) Reduce request frequency, 3) Consider upgrading plan",
        "E003": "Invalid request format. Solutions: 1) Check JSON syntax, 2) Verify required fields present, 3) Validate data types",
        "E004": "Resource not found. Solutions: 1) Verify resource ID, 2) Check resource hasn't been deleted, 3) Confirm correct environment (prod/staging)",
        "E005": "Internal server error. Solutions: 1) Retry after 30 seconds, 2) Check system status page, 3) Contact support if persists"
    }
    code_upper = code.upper().strip()
    return errors.get(code_upper, f"Unknown error code: {code}. Common codes: E001-E005")


def create_support_agent(system_prompt: str):
    """Create a support agent with the given system prompt."""
    model = ChatOpenAI(model="gpt-4o", temperature=0.1)
    agent = create_agent(model=model, tools=[check_system_status, lookup_error_code])
    return agent, system_prompt


def query_agent(agent, system_prompt: str, question: str) -> str:
    """Ask the agent a question and return the response."""
    result = agent.invoke({
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    })
    return result['messages'][-1].content


print("Support agent components ready!")

Support agent components ready!


In [3]:
# Current system prompt - intentionally basic
BASELINE_PROMPT = "You are a technical support assistant. Help users with their issues."

# Multiple feedback items collected over a week of production usage
feedback_collection = [
    {
        "query": "I keep getting E001 errors, what's wrong?",
        "response": "E001 means authentication failed. Check your API key.",
        "rating": 2,
        "comments": "Too brief! I already knew it was auth related. I needed step-by-step help to debug this. Also felt like talking to a robot.",
        "user_id": "user_042"
    },
    {
        "query": "Is the API down? My requests are timing out",
        "response": "The API is operational with 145ms response time.",
        "rating": 1,
        "comments": "This didn't help at all. My requests ARE timing out even if the status says operational. The agent didn't try to understand my actual problem or ask follow-up questions.",
        "user_id": "user_107"
    },
    {
        "query": "Getting error E002, how do I fix it?",
        "response": "E002 is rate limit exceeded. Implement exponential backoff, reduce request frequency, or upgrade your plan.",
        "rating": 3,
        "comments": "The information was correct but felt like a copy-paste from docs. Would have been nice if it asked about my use case to give more specific advice.",
        "user_id": "user_203"
    },
    {
        "query": "Auth service seems slow today, is there an issue?",
        "response": "The auth service is currently degraded with intermittent delays in token refresh.",
        "rating": 2,
        "comments": "OK so there IS an issue - but what should I do about it? Any workaround? When will it be fixed? The agent just stated facts without being helpful.",
        "user_id": "user_089"
    },
]

print(f"Baseline prompt: '{BASELINE_PROMPT}'")
print(f"\nCollected {len(feedback_collection)} feedback items:")
print("-" * 60)
for i, fb in enumerate(feedback_collection, 1):
    print(f"\n[{i}] Rating: {fb['rating']}/5")
    print(f"    Query: {fb['query']}")
    print(f"    Feedback: {fb['comments'][:80]}...")

Baseline prompt: 'You are a technical support assistant. Help users with their issues.'

Collected 4 feedback items:
------------------------------------------------------------

[1] Rating: 2/5
    Query: I keep getting E001 errors, what's wrong?
    Feedback: Too brief! I already knew it was auth related. I needed step-by-step help to deb...

[2] Rating: 1/5
    Query: Is the API down? My requests are timing out
    Feedback: This didn't help at all. My requests ARE timing out even if the status says oper...

[3] Rating: 3/5
    Query: Getting error E002, how do I fix it?
    Feedback: The information was correct but felt like a copy-paste from docs. Would have bee...

[4] Rating: 2/5
    Query: Auth service seems slow today, is there an issue?
    Feedback: OK so there IS an issue - but what should I do about it? Any workaround? When wi...


## Step 1: Analyze Feedback Themes

Before building a meta-prompt, you need to identify what's actually wrong. Look across all feedback items and extract the **common themes** - issues that appear in multiple pieces of feedback.

**Hint:** Consider categories like:
- Tone/personality issues
- Missing proactive behaviors  
- Lack of depth or context
- Failure to understand user needs

In [ ]:
def analyze_feedback_themes(feedback_items: List[Dict[str, Any]]) -> List[str]:
    """
    Analyze multiple feedback items to identify common themes/patterns.
    
    Look across all feedback comments and identify recurring issues that
    appear in 2 or more feedback items. These are the systemic problems
    that need to be addressed in the improved prompt.
    
    Args:
        feedback_items: List of feedback dictionaries, each containing:
            - query: User's original question
            - response: Agent's response  
            - rating: Numeric rating (1-5)
            - comments: User's detailed feedback
    
    Returns:
        List of 3-5 theme strings describing common issues found.
        Each theme should be a clear, actionable description.
        
        Example output:
        [
            "Responses lack empathy and feel robotic",
            "Agent doesn't ask clarifying questions",
            "Solutions are generic, not tailored to user's situation"
        ]
    """
    # TODO: Analyze the feedback items and identify common themes
    # Look for patterns that appear across multiple feedback items
    # Return a list of theme strings
    pass

## Step 2: Build the Synthesis Meta-Prompt

Now create a meta-prompt that incorporates your identified themes. Unlike the guided practice which used a single feedback item, your meta-prompt should present the **synthesized themes** to the prompt-engineer LLM.

**Key difference from guided practice:** You're not passing raw feedback - you're passing your analysis of what's wrong across multiple interactions.

In [ ]:
def build_synthesis_prompt(
    current_prompt: str, 
    themes: List[str], 
    sample_interactions: List[Dict[str, Any]]
) -> str:
    """
    Build a meta-prompt that synthesizes identified themes into prompt improvements.
    
    This meta-prompt should instruct an LLM to generate an improved system prompt
    that addresses ALL the identified themes, not just individual feedback items.
    
    Args:
        current_prompt: The existing system prompt
        themes: List of common issue themes identified from feedback analysis
        sample_interactions: 2-3 example interactions to illustrate the problems
    
    Returns:
        A meta-prompt string that:
        - Presents the current prompt
        - Lists the synthesized themes as systemic issues to address
        - Includes a few sample interactions as concrete examples
        - Instructs the LLM to generate an improved prompt addressing ALL themes
        - Specifies output format (prompt text only, no explanations)
    """
    # TODO: Build the synthesis meta-prompt
    # Structure it to focus on themes rather than individual feedback
    pass

## Step 3: Generate the Improved Prompt

Use your synthesis meta-prompt with an LLM to generate the improved system prompt.

In [ ]:
def generate_improved_prompt(
    current_prompt: str,
    feedback_items: List[Dict[str, Any]]
) -> str:
    """
    Complete pipeline: analyze feedback, build synthesis prompt, generate improvement.
    
    This function orchestrates the full improvement process:
    1. Call analyze_feedback_themes() to identify common issues
    2. Select 2-3 representative interactions as examples
    3. Call build_synthesis_prompt() to create the meta-prompt
    4. Send to an LLM and return the improved prompt
    
    Args:
        current_prompt: The existing system prompt
        feedback_items: List of all feedback dictionaries
        
    Returns:
        The improved system prompt generated by the LLM
    """
    # TODO: Implement the full pipeline
    # 1. Analyze themes
    # 2. Select sample interactions (pick 2-3 with lowest ratings)
    # 3. Build synthesis prompt
    # 4. Call LLM (use temperature=0.7) and return result
    pass

## Run Your Implementation

Execute the full pipeline and examine the results.

In [ ]:
# First, let's see your theme analysis
print("STEP 1: Analyzing feedback themes...")
print("=" * 60)

themes = analyze_feedback_themes(feedback_collection)

print("\nIdentified themes:")
for i, theme in enumerate(themes, 1):
    print(f"  {i}. {theme}")

print("\n" + "=" * 60)
print("STEP 2 & 3: Generating improved prompt...")
print("=" * 60)

improved_prompt = generate_improved_prompt(BASELINE_PROMPT, feedback_collection)

print("\nBASELINE PROMPT:")
print("-" * 60)
print(BASELINE_PROMPT)

print("\n\nIMPROVED PROMPT:")
print("-" * 60)
print(improved_prompt)
print("=" * 60)

## Test the Improvement

Compare how the baseline and improved agents handle similar queries.

In [ ]:
# Create both agents
baseline_agent, baseline_sys = create_support_agent(BASELINE_PROMPT)
improved_agent, improved_sys = create_support_agent(improved_prompt)

# Test with a query similar to the problematic ones
test_question = "I'm getting E003 errors intermittently. What's going on?"

print(f"Test Question: {test_question}")
print("=" * 70)

print("\nBASELINE AGENT RESPONSE:")
print("-" * 70)
baseline_response = query_agent(baseline_agent, baseline_sys, test_question)
print(baseline_response)

print("\n\nIMPROVED AGENT RESPONSE:")
print("-" * 70)
improved_response = query_agent(improved_agent, improved_sys, test_question)
print(improved_response)

print("\n" + "=" * 70)
print("SUCCESS CHECK:")
print("The improved response should address the identified themes:")
for theme in themes:
    print(f"  - {theme}")
print("=" * 70)